In [1]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
import os
import sys

sys.path.insert(0, os.path.realpath(os.path.join(os.getcwd(), "..")))
import getdist
import IPython
import matplotlib
import matplotlib.pyplot as plt
from getdist import MCSamples, plots

print("GetDist Version: %s, Matplotlib version: %s" % (getdist.__version__, matplotlib.__version__))

GetDist Version: 1.7.4, Matplotlib version: 3.10.0


In [2]:
import pandas as pd
from getdist import MCSamples, plots
import numpy as np

In [4]:
om = 0.3
ob = 0.049
h = 0.7
s = 44.5 * (np.log(9.83/(om*h**2.))/np.sqrt(1.+10.*(ob*h**2.)**0.75))
print(s)

147.39698390707758


In [ ]:
# --- LOAD TURING DATA ---
# Read the CSVs exported from Julia
# lcdm = pd.read_csv("lcdm_chains_tot.csv")
# gp = pd.read_csv("gp_chains_tot.csv")
lcdm = pd.read_csv("lcdm_chain_finaltest_-cmb.csv")
gp = pd.read_csv("gp_chain_finaltest.csv")

lcdm_omegab = pd.read_csv("lcdm_chains_normal_+omegab.csv")


# # Convert to numpy arrays
samps_lcdm = np.array(lcdm[["omegam_pr", "s8_pr", "h0_pr", "rd_pr", "M_pr"]])
samps_gp = np.array(gp[["omegam_po", "s8_po", "h_po", "r_po", "M_po"]])  
samps_lcdm_omegab = np.array(lcdm_omegab[["omegam_pr", "s8_pr", "h0_pr", "omegab_pr"]])

# --- CREATE GETDIST OBJECTS ---
names = ["om", "s8", "h0", "R", "M"]
labels = [r"\Omega_m", r"\sigma_8", r"h_0", r"R_d", r"M"]
names2 = ["om", "s8", "h0", "omegab"]
labels2 = [r"\Omega_m", r"\sigma_8", r"h_0", r"\Omega_b"]

samples1 = MCSamples(samples=samps_lcdm, names=names, labels=labels, label="LCDM")
samples2 = MCSamples(samples=samps_gp, names=names, labels=labels, label="GP")
samples3 = MCSamples(samples=samps_lcdm_omegab, names=names2, labels=labels2, label="LCDM + Omega_b")


In [ ]:
# 1D marginalized plot
g = plots.get_single_plotter(width_inch=4)
g.plot_1d(samples1, "h0")
g.fig

In [ ]:
from getdist import plots
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# --- Planck values ---
means = {"om": 0.317, "h0": 0.6736, "s8": 0.812, "M": -19.3}
sigmas = {"om": 0.008, "h0": 0.0054, "s8": 0.007, "M": 0.1}

params = ["om", "s8", "h0", "R", "M"]

# --- Triangle plot ---
h = plots.get_subplot_plotter()

h.triangle_plot(
    [samples1, samples2],
    params,
    filled=True,
    legend_labels=[r"$\Lambda$CDM", "GP"],
    legend_loc="upper right",
    line_args=[
        {"lw": 1.5, "color": "blue"},
        {"lw": 1.5, "color": "green"}
    ],
    contour_colors=["blue", "green"],
    markers=means
)

# --- Add Planck uncertainty bands (diagonal plots only) ---
for i, p in enumerate(params):
    if p in means:
        ax = h.subplots[i, i]  # diagonal subplot
        
        mu = means[p]
        sigma = sigmas[p]
        
        ax.axvspan(
            mu - sigma,
            mu + sigma,
            color="gray",
            alpha=0.3,
            zorder=0  # behind contours
        )


# Planck patch
planck_patch = mpatches.Patch(color='gray', alpha=0.3, label='Planck 2018')

# Get current legend (created by getdist)
leg = h.fig.legends[0]

# Extract existing handles + labels
handles = leg.legend_handles
labels = [t.get_text() for t in leg.texts]

# Add Planck
handles.append(planck_patch)
labels.append("Planck 2018")

# Remove old legend
leg.remove()

# Add updated legend
h.fig.legend(handles, labels, loc="upper right")

# --- Show figure ---
plt.savefig("cornerplot_minusmu.pdf",
            format="pdf",
            bbox_inches="tight")
plt.show()

In [ ]:
fs8df = pd.read_csv("fs8_data.csv")
fs8planck = pd.read_csv("fs8_lcdm.csv")
fs8_data = fs8df["data"]
fs8_z = fs8df["z"]
fs8_error = fs8df["err"]
fs8_planck = fs8planck["LCDM"]
z_integ = pd.read_csv("zinteg1.csv")
# 1. Define the specific columns you want to include in the subset
# This creates a list: ['model[1]', 'model[2]', ..., 'model[20]']
fs8_target_cols1 = [f'fss8_model[{i}]' for i in range(1, 301)]
fs8_target_cols2 = [f'fss8_mod[{i}]' for i in range(1, 301)]

# 2. Extract the subset and calculate statistics
fs8_lcdm_df = lcdm[fs8_target_cols1]
fs8_gp_df = gp[fs8_target_cols2]

fs8_mean_vals_lcdm = fs8_lcdm_df.mean()
fs8_std_vals_lcdm = fs8_lcdm_df.std()

fs8_mean_vals_gp = fs8_gp_df.mean()
fs8_std_vals_gp = fs8_gp_df.std()

# 3. Create the plot
# z = z_integ["z"]  # X-axis representing the models 1 to 20
# plt.figure(figsize=(10, 6))

# #fs8_data
# plt.errorbar(fs8_z[1:len(fs8_z)], fs8_data[1:len(fs8_data)], yerr=fs8_error[1:len(fs8_error)], fmt='.', color='black', capsize=2, label=r'$f\sigma_8$ data - DESI')
# # plt.errorbar(fs8_z[0], fs8_data[0], yerr=fs8_error[0], fmt='.', color='green', capsize=2, label=r'$f\sigma_8$ data - 6dFGS')

# # Plot LCDM
# plt.plot(z, fs8_mean_vals_lcdm, color='blue', label='LCDM mean', linewidth=2)
# plt.fill_between(z, fs8_mean_vals_lcdm - fs8_std_vals_lcdm, fs8_mean_vals_lcdm + fs8_std_vals_lcdm, color='blue', alpha=0.3, label='LCDM standard dev.')

# #Plot GP
# plt.plot(z, fs8_mean_vals_gp, color='green', label='GP mean', linewidth=2)
# plt.fill_between(z, fs8_mean_vals_gp - fs8_std_vals_gp, fs8_mean_vals_gp + fs8_std_vals_gp, color='green', alpha=0.3, label='GP standard dev.')

# #plot planck
# plt.plot(z, fs8_planck, color='black', linestyle='--', label='Planck 2018')

# # Formatting
# plt.title(r'Comparison of LCDM and GP constraints on $f\sigma_8$')
# plt.xlabel(r'$z$')
# plt.ylabel(r'$f\sigma_8$')
# # plt.xticks(z)
# plt.legend()
# # plt.grid(True, linestyle=':', alpha=0.7)

# plt.show()


In [ ]:
hzdf = pd.read_csv("hz_data.csv")
hzplanck = pd.read_csv("hz_lcdm.csv")
hz_data = hzdf["data"]
hz_z = hzdf["z"]
hz_error = hzdf["err"]
hz_planck = hzplanck["LCDM"]
z_integ = pd.read_csv("z_integ1.csv")
# 1. Define the specific columns you want to include in the subset
# This creates a list: ['model[1]', 'model[2]', ..., 'model[20]']
hz_target_cols1 = [f'hz_model[{i}]' for i in range(1, 301)]
hz_target_cols2 = [f'hz_mod[{i}]' for i in range(1, 301)]

# 2. Extract the subset and calculate statistics
hz_lcdm_df = lcdm[hz_target_cols1]
hz_gp_df = gp[hz_target_cols2]

hz_mean_vals_lcdm = hz_lcdm_df.mean()
hz_std_vals_lcdm = hz_lcdm_df.std()

hz_mean_vals_gp = hz_gp_df.mean()
hz_std_vals_gp = hz_gp_df.std()

# 3. Create the plot
print(z_integ["z"])
z = z_integ["z"]  # X-axis representing the models 1 to 20
# plt.figure(figsize=(10, 6))

# #hz_data
# plt.errorbar(hz_z, hz_data, yerr=hz_error, fmt='.', color='black', capsize=2, label='H(z) data - cosmic chronometers')

# # Plot LCDM
# plt.plot(z, hz_mean_vals_lcdm, color='blue', label='LCDM mean', linewidth=2)
# plt.fill_between(z, hz_mean_vals_lcdm - hz_std_vals_lcdm, hz_mean_vals_lcdm + hz_std_vals_lcdm, color='blue', alpha=0.3, label='LCDM standard dev.')

# #Plot GP
# plt.plot(z, hz_mean_vals_gp, color='green', label='GP mean', linewidth=2)
# plt.fill_between(z, hz_mean_vals_gp - hz_std_vals_gp, hz_mean_vals_gp + hz_std_vals_gp, color='green', alpha=0.3, label='GP standard dev.')

# #plot planck
# plt.plot(z, hz_planck, color='black', linestyle='--', label='Planck 2018')

# # Formatting
# plt.title('Comparison of LCDM and GP constraints on H(z)')
# plt.xlabel(r'$z$')
# plt.ylabel(r'$H(z) \ [\mathrm{km \ s^{-1} \ Mpc^{-1}}]$')
# # plt.xticks(z)
# plt.legend()
# # plt.grid(True, linestyle=':', alpha=0.7)

# plt.show()

In [ ]:
# omegam
# This creates a list: ['model[1]', 'model[2]', ..., 'model[20]']
om_lcdm = [f'omegam_pr']
om_gp = [f'omegam_po']

# 2. Extract the subset and calculate statistics
lcdm_om = lcdm[om_lcdm]
gp_om = gp[om_gp]

mean_vals_lcdm_om = lcdm_om.mean()
std_vals_lcdm_om = lcdm_om.std()

mean_vals_gp_om = gp_om.mean()
std_vals_gp_om = gp_om.std()

# s8
# This creates a list: ['model[1]', 'model[2]', ..., 'model[20]']
s8_lcdm = [f's8_pr']
s8_gp = [f's8_po']

# 2. Extract the subset and calculate statistics
lcdm_s8 = lcdm[s8_lcdm]
gp_s8 = gp[s8_gp]

mean_vals_lcdm_s8 = lcdm_s8.mean()
std_vals_lcdm_s8 = lcdm_s8.std()

mean_vals_gp_s8 = gp_s8.mean()
std_vals_gp_s8 = gp_s8.std()

# h0
# This creates a list: ['model[1]', 'model[2]', ..., 'model[20]']
h0_lcdm = [f'h0_pr']
h0_gp = [f'h_po']

# 2. Extract the subset and calculate statistics
lcdm_h0 = lcdm[h0_lcdm]
gp_h0 = gp[h0_gp]

mean_vals_lcdm_h0 = lcdm_h0.mean()
std_vals_lcdm_h0 = lcdm_h0.std()

mean_vals_gp_h0 = gp_h0.mean()
std_vals_gp_h0 = gp_h0.std()

print(f"LCDM Omega_m: {mean_vals_lcdm_om.values[0]:.4f} ± {std_vals_lcdm_om.values[0]:.4f}")
print(f"GP Omega_m: {mean_vals_gp_om.values[0]:.4f} ± {std_vals_gp_om.values[0]:.4f}")

print(f"LCDM Sigma_8: {mean_vals_lcdm_s8.values[0]:.4f} ± {std_vals_lcdm_s8.values[0]:.4f}")
print(f"GP Sigma_8: {mean_vals_gp_s8.values[0]:.4f} ± {std_vals_gp_s8.values[0]:.4f}")

print(f"LCDM h0: {mean_vals_lcdm_h0.values[0]:.4f} ± {std_vals_lcdm_h0.values[0]:.4f}")
print(f"GP h0: {mean_vals_gp_h0.values[0]:.4f} ± {std_vals_gp_h0.values[0]:.4f}")

In [ ]:
# rd
# This creates a list: ['model[1]', 'model[2]', ..., 'model[20]']
rd_lcdm = [f'rd_pr']
rd_gp = [f'r_po']

# 2. Extract the subset and calculate statistics
lcdm_rd = lcdm[rd_lcdm]
gp_rd = gp[rd_gp]

mean_vals_lcdm_rd = lcdm_rd.mean()
std_vals_lcdm_rd = lcdm_rd.std()

mean_vals_gp_rd = gp_rd.mean()
std_vals_gp_rd = gp_rd.std()

# 2. Extract the subset and calculate statistics

print(f"LCDM rd: {mean_vals_lcdm_rd.values[0]:.4f} ± {std_vals_lcdm_rd.values[0]:.4f}")
print(f"GP rd: {mean_vals_gp_rd.values[0]:.4f} ± {std_vals_gp_rd.values[0]:.4f}")


In [ ]:
# Relative standard deviation hz
rsd_pct_lcdm_hz = (hz_std_vals_lcdm / hz_mean_vals_lcdm) * 100  # use abs to avoid negative means
rsd_pct_gp_hz = (hz_std_vals_gp / hz_mean_vals_gp) * 100  # use abs to avoid negative means

rsd_pct_lcdm_fs8 = (fs8_std_vals_lcdm / fs8_mean_vals_lcdm) * 100
rsd_pct_gp_fs8 = (fs8_std_vals_gp / fs8_mean_vals_gp) * 100


In [ ]:
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

fig, ax = plt.subplots(figsize=(10, 6))

# --- Main plot ---
ax.errorbar(hz_z, hz_data, yerr=hz_error, fmt='.', capsize=2, color='black',
            label='H(z) data - cosmic chronometers')

# LCDM → BLUE
print(z)
ax.plot(z, hz_mean_vals_lcdm, linewidth=2, color='blue', label=r'$\Lambda$CDM')
ax.fill_between(z,
                hz_mean_vals_lcdm - hz_std_vals_lcdm,
                hz_mean_vals_lcdm + hz_std_vals_lcdm,
                color='blue', alpha=0.3)

# GP → GREEN
ax.plot(z, hz_mean_vals_gp, linewidth=2, color='green', label='GP')
ax.fill_between(z,
                hz_mean_vals_gp - hz_std_vals_gp,
                hz_mean_vals_gp + hz_std_vals_gp,
                color='green', alpha=0.3)

ax.plot(z, hz_planck, linestyle='--', label='Planck 2018', color='black')

ax.set_title('Comparison of LCDM and GP constraints on H(z)')
ax.set_xlabel(r'$z$')
ax.set_ylabel(r'$H(z) \ [\mathrm{km \ s^{-1} \ Mpc^{-1}}]$')
ax.legend(fontsize=8)

# --- Inset (subpanel) ---
ax_inset = ax.inset_axes([0.62, 0.08, 0.35, 0.35])

ax_inset.plot(z, rsd_pct_lcdm_hz, marker='.', linestyle='', markersize=0.8,
              color='blue', label='LCDM')
ax_inset.plot(z, rsd_pct_gp_hz, marker='.', linestyle='', markersize=0.8,
              color='green', label='GP')

ax_inset.set_xlabel("z", fontsize=6)
ax_inset.set_ylabel("Relative standard deviation (%)", fontsize=6)
ax_inset.tick_params(axis='both', labelsize=6)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# --- Main plot ---
ax.errorbar(fs8_z, fs8_data, yerr=fs8_error, fmt='.', capsize=2, color='black',
            label='fσ₈ data - DESI')

# LCDM → BLUE
ax.plot(z, fs8_mean_vals_lcdm, linewidth=2, color='blue', label=r'$\Lambda$CDM')
ax.fill_between(z,
                fs8_mean_vals_lcdm - fs8_std_vals_lcdm,
                fs8_mean_vals_lcdm + fs8_std_vals_lcdm,
                color='blue', alpha=0.3)

# GP → GREEN
ax.plot(z, fs8_mean_vals_gp, linewidth=2, color='green', label='GP')
ax.fill_between(z,
                fs8_mean_vals_gp - fs8_std_vals_gp,
                fs8_mean_vals_gp + fs8_std_vals_gp,
                color='green', alpha=0.3)

ax.plot(z, fs8_planck, linestyle='--', label='Planck 2018', color='black')

ax.set_title('Comparison of LCDM and GP constraints on fσ₈')
ax.set_xlabel(r'$z$')
ax.set_ylabel(r'$f\sigma_8$')
ax.legend(fontsize=8, loc='upper left')

# --- Inset (subpanel) ---
ax_inset = ax.inset_axes([0.62, 0.62, 0.35, 0.35])

ax_inset.plot(z, rsd_pct_lcdm_fs8, marker='.', linestyle='', markersize=0.8,
              color='blue', label='LCDM')
ax_inset.plot(z, rsd_pct_gp_fs8, marker='.', linestyle='', markersize=0.8,
              color='green', label='GP')

ax_inset.set_xlabel("z", fontsize=6)
ax_inset.set_ylabel("Relative standard deviation (%)", fontsize=6)
ax_inset.tick_params(axis='both', labelsize=6)

plt.show()

In [ ]:

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(20, 20), sharex=True)

# =========================
# 🔹 TOP PANEL: H(z)
# =========================
ax1.errorbar(hz_z, hz_data, yerr=hz_error, fmt='.', capsize=2, color='black',
             label='H(z) data - cosmic chronometers')

ax1.plot(z, hz_mean_vals_lcdm, linewidth=2, color='blue', label=r'$\Lambda$CDM')
ax1.fill_between(z,
                 hz_mean_vals_lcdm - hz_std_vals_lcdm,
                 hz_mean_vals_lcdm + hz_std_vals_lcdm,
                 color='blue', alpha=0.3)

ax1.plot(z, hz_mean_vals_gp, linewidth=2, color='green', label='GP')
ax1.fill_between(z,
                 hz_mean_vals_gp - hz_std_vals_gp,
                 hz_mean_vals_gp + hz_std_vals_gp,
                 color='green', alpha=0.3)

ax1.plot(z, hz_planck, linestyle='--', color='black', label='Planck 2018')

ax1.set_ylabel(r'$H(z)\ [\mathrm{km \ s^{-1} \ Mpc^{-1}}]$', fontsize=18)
ax1.set_title('Comparison of LCDM and GP constraints', fontsize=18)
ax1.tick_params(axis='both', labelsize=18)
ax1.legend(fontsize=18)

# Inset for H(z)
ax1_inset = ax1.inset_axes([0.63, 0.08, 0.35, 0.35])
ax1_inset.plot(z, rsd_pct_lcdm_hz, '.', markersize=0.8, color='blue')
ax1_inset.plot(z, rsd_pct_gp_hz, '.', markersize=0.8, color='green')
ax1_inset.set_xlabel("z", fontsize=15)
ax1_inset.set_ylabel("Rel. std (%)", fontsize=15)
ax1_inset.tick_params(axis='both', labelsize=15)

# =========================
# 🔹 BOTTOM PANEL: fσ₈
# =========================
ax2.errorbar(fs8_z, fs8_data, yerr=fs8_error, fmt='.', capsize=2, color='black',
             label='fσ₈ data - DESI')

ax2.plot(z, fs8_mean_vals_lcdm, linewidth=2, color='blue', label=r'$\Lambda$CDM')
ax2.fill_between(z,
                 fs8_mean_vals_lcdm - fs8_std_vals_lcdm,
                 fs8_mean_vals_lcdm + fs8_std_vals_lcdm,
                 color='blue', alpha=0.3)

ax2.plot(z, fs8_mean_vals_gp, linewidth=2, color='green', label='GP')
ax2.fill_between(z,
                 fs8_mean_vals_gp - fs8_std_vals_gp,
                 fs8_mean_vals_gp + fs8_std_vals_gp,
                 color='green', alpha=0.3)

ax2.plot(z, fs8_planck, linestyle='--', color='black', label='Planck 2018')

ax2.set_xlabel(r'$z$', fontsize=18)
ax2.set_ylabel(r'$f\sigma_8$', fontsize=18)
ax2.tick_params(axis='both', labelsize=18)
ax2.legend(fontsize=18, loc='upper left')

# Inset for fσ₈
ax2_inset = ax2.inset_axes([0.63, 0.63, 0.35, 0.35])
ax2_inset.plot(z, rsd_pct_lcdm_fs8, '.', markersize=0.8, color='blue')
ax2_inset.plot(z, rsd_pct_gp_fs8, '.', markersize=0.8, color='green')
ax2_inset.set_xlabel("z", fontsize=15)
ax2_inset.set_ylabel("Rel. std (%)", fontsize=15)
ax2_inset.tick_params(axis='both', labelsize=15)

# =========================
# Layout fix
# =========================
plt.subplots_adjust(hspace=0.1)
plt.tight_layout()
plt.savefig("minusmu_hz_fs8.pdf",
            format="pdf",
            bbox_inches="tight")
plt.show()

In [ ]:


fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 14), sharex=True)

# =========================
# 🔹 TOP PANEL: H(z)
# =========================
ax1.errorbar(
    hz_z, hz_data, yerr=hz_error,
    fmt='o', markersize=6,
    markerfacecolor='black', markeredgecolor='black',
    ecolor='black', elinewidth=1, capsize=2,
    linestyle='none',
    label='Cosmic chronometers'
)

ax1.plot(z, hz_mean_vals_lcdm, linewidth=2, color='blue', label=r'$\Lambda$CDM')
ax1.fill_between(
    z,
    hz_mean_vals_lcdm - hz_std_vals_lcdm,
    hz_mean_vals_lcdm + hz_std_vals_lcdm,
    color='blue', alpha=0.2
)

ax1.plot(z, hz_mean_vals_gp, linewidth=2, color='green', label='GP')
ax1.fill_between(
    z,
    hz_mean_vals_gp - hz_std_vals_gp,
    hz_mean_vals_gp + hz_std_vals_gp,
    color='green', alpha=0.2
)

ax1.plot(z, hz_planck, linestyle='--', linewidth=1.5, color='black', label='Planck 2018')

ax1.set_ylabel(r'$H(z)\ [\mathrm{km \ s^{-1} \ Mpc^{-1}}]$', fontsize=16)
ax1.set_title('Constraints on $H(z)$ and $f\\sigma_8$', fontsize=18)

ax1.tick_params(axis='both', labelsize=14)
ax1.legend(fontsize=12, frameon=False)

# Inset
ax1_inset = ax1.inset_axes([0.60, 0.08, 0.35, 0.35])
ax1_inset.plot(z, rsd_pct_lcdm_hz, 'o', markersize=2, color='blue')
ax1_inset.plot(z, rsd_pct_gp_hz, 'o', markersize=2, color='green')
ax1_inset.set_xlabel("z", fontsize=10)
ax1_inset.set_ylabel("Rel. std (%)", fontsize=10)
ax1_inset.tick_params(axis='both', labelsize=9)

# =========================
# 🔹 BOTTOM PANEL: fσ₈
# =========================
ax2.errorbar(
    fs8_z, fs8_data, yerr=fs8_error,
    fmt='o', markersize=6,
    markerfacecolor='black', markeredgecolor='black',
    ecolor='black', elinewidth=1, capsize=2,
    linestyle='none',
    label='DESI'
)

ax2.plot(z, fs8_mean_vals_lcdm, linewidth=2, color='blue', label=r'$\Lambda$CDM')
ax2.fill_between(
    z,
    fs8_mean_vals_lcdm - fs8_std_vals_lcdm,
    fs8_mean_vals_lcdm + fs8_std_vals_lcdm,
    color='blue', alpha=0.2
)

ax2.plot(z, fs8_mean_vals_gp, linewidth=2, color='green', label='GP')
ax2.fill_between(
    z,
    fs8_mean_vals_gp - fs8_std_vals_gp,
    fs8_mean_vals_gp + fs8_std_vals_gp,
    color='green', alpha=0.2
)

ax2.plot(z, fs8_planck, linestyle='--', linewidth=1.5, color='black', label='Planck 2018')

ax2.set_xlabel(r'$z$', fontsize=16)
ax2.set_ylabel(r'$f\sigma_8$', fontsize=16)

ax2.tick_params(axis='both', labelsize=14)
ax2.legend(fontsize=12, loc='upper left', frameon=False)

# Inset
ax2_inset = ax2.inset_axes([0.60, 0.60, 0.35, 0.35])
ax2_inset.plot(z, rsd_pct_lcdm_fs8, 'o', markersize=2, color='blue')
ax2_inset.plot(z, rsd_pct_gp_fs8, 'o', markersize=2, color='green')
ax2_inset.set_xlabel("z", fontsize=10)
ax2_inset.set_ylabel("Rel. std (%)", fontsize=10)
ax2_inset.tick_params(axis='both', labelsize=9)

# =========================
# Layout
# =========================
plt.subplots_adjust(hspace=0.08)

plt.savefig(
    "minusmu_hz_fs8.pdf",
    format="pdf",
    bbox_inches="tight"
)

plt.show()